# Reusable EDA Report Functions

Use `run_eda_report(df, dataset_name="...", target_col="...")` to generate a repeatable EDA workflow for future datasets.

In [ ]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = print


def show_title(title: str, level: int = 2) -> None:
    """Display a section title in notebooks, or print it in scripts."""
    if Markdown is not None:
        display(Markdown(f"{'#' * level} {title}"))
    else:
        print("\n" + title.upper())


def show_table(obj) -> None:
    """Display a dataframe/series nicely in notebooks."""
    display(obj)


def get_feature_types(df: pd.DataFrame) -> dict:
    """Return numeric, categorical, datetime, and boolean feature lists."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
    bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
    categorical_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()

    return {
        "numeric": numeric_cols,
        "categorical": categorical_cols,
        "datetime": datetime_cols,
        "boolean": bool_cols,
    }


def dataset_information(
    df: pd.DataFrame,
    dataset_name: str = "Dataset",
    date_col: Optional[str] = None,
) -> pd.DataFrame:
    """Create the dataset information section."""
    show_title("Dataset Information")

    start_date = end_date = "Not applicable"
    if date_col and date_col in df.columns:
        dates = pd.to_datetime(df[date_col], errors="coerce")
        if dates.notna().any():
            start_date = dates.min()
            end_date = dates.max()

    info = pd.DataFrame(
        {
            "Metric": ["Dataset Name", "Number of Rows", "Number of Columns", "Date Range"],
            "Value": [dataset_name, df.shape[0], df.shape[1], f"{start_date} to {end_date}"],
        }
    )
    show_table(info)
    return info


def data_overview(df: pd.DataFrame, sample_size: int = 5) -> dict:
    """Show shape, dtypes, duplicate count, and sample rows."""
    show_title("Data Overview")
    feature_types = get_feature_types(df)

    overview = pd.DataFrame(
        {
            "Metric": [
                "Rows",
                "Columns",
                "Duplicate Rows",
                "Numeric Features",
                "Categorical Features",
                "Datetime Features",
            ],
            "Value": [
                df.shape[0],
                df.shape[1],
                df.duplicated().sum(),
                len(feature_types["numeric"]),
                len(feature_types["categorical"]),
                len(feature_types["datetime"]),
            ],
        }
    )

    show_table(overview)
    show_title("Column Types", level=3)
    show_table(df.dtypes.astype(str).to_frame("dtype"))
    show_title("Sample Rows", level=3)
    show_table(df.head(sample_size))

    return {"overview": overview, "feature_types": feature_types}


def data_loading_and_cleaning(
    df: pd.DataFrame,
    drop_duplicates: bool = False,
    standardize_column_names: bool = False,
) -> pd.DataFrame:
    """Optionally clean a copy of the dataset."""
    show_title("Data Loading and Cleaning")
    cleaned = df.copy()

    if standardize_column_names:
        cleaned.columns = (
            cleaned.columns.astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r"\s+", "_", regex=True)
        )

    before_rows = len(cleaned)
    if drop_duplicates:
        cleaned = cleaned.drop_duplicates()

    cleaning_summary = pd.DataFrame(
        {
            "Step": ["Standardized column names", "Dropped duplicate rows", "Rows before", "Rows after"],
            "Value": [standardize_column_names, drop_duplicates, before_rows, len(cleaned)],
        }
    )
    show_table(cleaning_summary)
    return cleaned


def summary_statistics(df: pd.DataFrame) -> dict:
    """Show descriptive statistics for numeric and categorical features."""
    show_title("Data Summary and Descriptive Statistics")

    numeric_summary = df.describe(include=[np.number]).T
    categorical_summary = df.describe(include=["object", "category", "string", "bool"]).T

    if not numeric_summary.empty:
        show_title("Numeric Summary Statistics", level=3)
        show_table(numeric_summary)

    if not categorical_summary.empty:
        show_title("Categorical Summary Statistics", level=3)
        show_table(categorical_summary)

    return {"numeric": numeric_summary, "categorical": categorical_summary}


def missing_values_analysis(df: pd.DataFrame) -> pd.DataFrame:
    """Analyze missing values by column."""
    show_title("Missing Values Analysis")

    missing = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_percent": df.isna().mean().mul(100).round(2),
        }
    ).sort_values("missing_percent", ascending=False)

    show_table(missing[missing["missing_count"] > 0] if missing["missing_count"].sum() else missing.head(10))
    return missing


def plot_feature_distributions(
    df: pd.DataFrame,
    numeric_cols: Optional[list[str]] = None,
    categorical_cols: Optional[list[str]] = None,
    max_numeric: int = 8,
    max_categorical: int = 8,
    top_n: int = 10,
) -> None:
    """Visualize distributions for numeric and categorical features."""
    show_title("Feature Distribution")
    feature_types = get_feature_types(df)
    numeric_cols = numeric_cols or feature_types["numeric"][:max_numeric]
    categorical_cols = categorical_cols or feature_types["categorical"][:max_categorical]

    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        df[col].dropna().hist(ax=axes[0], bins=30, edgecolor="black")
        axes[0].set_title(f"Histogram: {col}")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Frequency")

        axes[1].boxplot(df[col].dropna(), vert=False)
        axes[1].set_title(f"Box Plot: {col}")
        axes[1].set_xlabel(col)
        plt.tight_layout()
        plt.show()

    for col in categorical_cols:
        counts = df[col].value_counts(dropna=False).head(top_n)
        ax = counts.sort_values().plot(kind="barh", figsize=(10, 4))
        ax.set_title(f"Frequency: {col}")
        ax.set_xlabel("Count")
        ax.set_ylabel(col)
        plt.tight_layout()
        plt.show()


def univariate_analysis(
    df: pd.DataFrame,
    numeric_cols: Optional[list[str]] = None,
    categorical_cols: Optional[list[str]] = None,
    top_n: int = 10,
) -> dict:
    """Create frequency tables and descriptive statistics for individual features."""
    show_title("Univariate Analysis")
    feature_types = get_feature_types(df)
    numeric_cols = numeric_cols or feature_types["numeric"]
    categorical_cols = categorical_cols or feature_types["categorical"]

    numeric_stats = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()
    categorical_freq = {
        col: df[col].value_counts(dropna=False).head(top_n) for col in categorical_cols
    }

    if not numeric_stats.empty:
        show_title("Numeric Features", level=3)
        show_table(numeric_stats)

    if categorical_freq:
        show_title("Categorical Features", level=3)
        for col, freq in categorical_freq.items():
            show_title(col, level=4)
            show_table(freq.to_frame("count"))

    return {"numeric_stats": numeric_stats, "categorical_freq": categorical_freq}


def bivariate_analysis(
    df: pd.DataFrame,
    target_col: Optional[str] = None,
    numeric_cols: Optional[list[str]] = None,
    categorical_cols: Optional[list[str]] = None,
    max_pairs: int = 5,
) -> None:
    """Plot numeric-numeric and categorical-numeric relationships."""
    show_title("Bivariate Analysis")
    feature_types = get_feature_types(df)
    numeric_cols = numeric_cols or feature_types["numeric"]
    categorical_cols = categorical_cols or feature_types["categorical"]

    show_title("Numeric-Numeric Relationships", level=3)
    numeric_pairs = list(zip(numeric_cols, numeric_cols[1:]))[:max_pairs]
    for x_col, y_col in numeric_pairs:
        ax = df.plot.scatter(x=x_col, y=y_col, figsize=(6, 4), alpha=0.6)
        ax.set_title(f"{x_col} vs {y_col}")
        plt.tight_layout()
        plt.show()

    if target_col and target_col in df.columns and numeric_cols:
        show_title("Target Relationships", level=3)
        if pd.api.types.is_numeric_dtype(df[target_col]):
            for col in [c for c in numeric_cols if c != target_col][:max_pairs]:
                ax = df.plot.scatter(x=col, y=target_col, figsize=(6, 4), alpha=0.6)
                ax.set_title(f"{col} vs {target_col}")
                plt.tight_layout()
                plt.show()
        else:
            for col in numeric_cols[:max_pairs]:
                groups = [group[col].dropna() for _, group in df.groupby(target_col, observed=False)]
                labels = [str(label) for label, _ in df.groupby(target_col, observed=False)]
                if groups:
                    plt.figure(figsize=(10, 4))
                    plt.boxplot(groups, labels=labels, vert=False)
                    plt.title(f"{col} by {target_col}")
                    plt.xlabel(col)
                    plt.tight_layout()
                    plt.show()

    if categorical_cols and numeric_cols:
        show_title("Categorical-Numeric Relationships", level=3)
        for cat_col in categorical_cols[:2]:
            for num_col in numeric_cols[:2]:
                grouped = df.groupby(cat_col, observed=False)[num_col].median().sort_values().tail(10)
                ax = grouped.plot(kind="barh", figsize=(10, 4))
                ax.set_title(f"Median {num_col} by {cat_col}")
                ax.set_xlabel(f"Median {num_col}")
                plt.tight_layout()
                plt.show()


def feature_engineering(
    df: pd.DataFrame,
    date_col: Optional[str] = None,
    height_col: str = "Height",
    weight_col: str = "Weight",
) -> pd.DataFrame:
    """Create common EDA features on a copy of the dataframe."""
    show_title("Feature Engineering")
    engineered = df.copy()
    created_features = []

    if height_col in engineered.columns and weight_col in engineered.columns:
        height = pd.to_numeric(engineered[height_col], errors="coerce")
        weight = pd.to_numeric(engineered[weight_col], errors="coerce")
        engineered["BMI"] = weight / (height ** 2)
        created_features.append("BMI = Weight / Height^2")

    if date_col and date_col in engineered.columns:
        dates = pd.to_datetime(engineered[date_col], errors="coerce")
        engineered[f"{date_col}_year"] = dates.dt.year
        engineered[f"{date_col}_month"] = dates.dt.month
        engineered[f"{date_col}_dayofweek"] = dates.dt.dayofweek
        created_features.extend([
            f"{date_col}_year",
            f"{date_col}_month",
            f"{date_col}_dayofweek",
        ])

    result = pd.DataFrame({"Created Feature": created_features or ["No automatic features created"]})
    show_table(result)
    return engineered


def outlier_detection_iqr(
    df: pd.DataFrame,
    numeric_cols: Optional[list[str]] = None,
    iqr_multiplier: float = 1.5,
) -> pd.DataFrame:
    """Detect numeric outliers using the IQR rule."""
    show_title("Outlier Detection")
    numeric_cols = numeric_cols or get_feature_types(df)["numeric"]
    rows = []

    for col in numeric_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - iqr_multiplier * iqr
        upper = q3 + iqr_multiplier * iqr
        mask = (df[col] < lower) | (df[col] > upper)
        rows.append(
            {
                "feature": col,
                "lower_bound": lower,
                "upper_bound": upper,
                "outlier_count": int(mask.sum()),
                "outlier_percent": round(mask.mean() * 100, 2),
            }
        )

    outliers = pd.DataFrame(rows).sort_values("outlier_count", ascending=False)
    show_table(outliers)
    return outliers


def correlation_analysis(
    df: pd.DataFrame,
    numeric_cols: Optional[list[str]] = None,
    method: str = "pearson",
) -> pd.DataFrame:
    """Compute and visualize numeric correlations."""
    show_title("Correlation Analysis")
    numeric_cols = numeric_cols or get_feature_types(df)["numeric"]

    if len(numeric_cols) < 2:
        print("Not enough numeric columns for correlation analysis.")
        return pd.DataFrame()

    corr = df[numeric_cols].corr(method=method)
    show_table(corr)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.index)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(corr.index)
    ax.set_title(f"{method.title()} Correlation Matrix")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    return corr


def custom_target_analysis(df: pd.DataFrame, target_col: Optional[str] = None) -> dict:
    """Run target-focused analysis when a target column is provided."""
    show_title("Custom Analyses")

    if not target_col or target_col not in df.columns:
        print("No target column provided. Add target_col='your_column' to run target analysis.")
        return {}

    result = {}
    target_counts = df[target_col].value_counts(dropna=False)
    result["target_counts"] = target_counts

    show_title(f"Target Distribution: {target_col}", level=3)
    show_table(target_counts.to_frame("count"))

    ax = target_counts.sort_values().plot(kind="barh", figsize=(10, 4))
    ax.set_title(f"Target Distribution: {target_col}")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()

    return result


def conclusions_and_recommendations(
    missing: pd.DataFrame,
    outliers: pd.DataFrame,
    corr: pd.DataFrame,
) -> list[str]:
    """Generate starter recommendations from EDA outputs."""
    show_title("Conclusions and Recommendations")
    recommendations = []

    high_missing = missing[missing["missing_percent"] > 20]
    if not high_missing.empty:
        recommendations.append("Review high-missing columns before modeling; consider imputation or removal.")
    else:
        recommendations.append("Missing values do not appear severe based on the 20% threshold.")

    high_outliers = outliers[outliers["outlier_percent"] > 5] if not outliers.empty else pd.DataFrame()
    if not high_outliers.empty:
        recommendations.append("Investigate numeric features with many IQR outliers before applying models sensitive to scale.")

    if not corr.empty:
        upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        strong_pairs = upper.stack()[lambda s: s > 0.8]
        if not strong_pairs.empty:
            recommendations.append("Check strongly correlated feature pairs for redundancy or multicollinearity.")

    for item in recommendations:
        print(f"- {item}")

    return recommendations


def interactive_eda_tools(df: pd.DataFrame, output_dir: str = "reports") -> dict:
    """Show optional commands for interactive EDA tools."""
    show_title("Interactive EDA Tools")
    output_path = Path(output_dir)

    tools = {
        "ydata-profiling": "pip install ydata-profiling",
        "sweetviz": "pip install sweetviz",
    }

    print("Optional tools you can install for interactive reports:")
    for tool, command in tools.items():
        print(f"- {tool}: {command}")
    print(f"Suggested output folder: {output_path.resolve()}")
    return tools


def next_steps() -> list[str]:
    """Print common next steps after EDA."""
    show_title("Next Steps")
    steps = [
        "Confirm the business question and target variable.",
        "Decide missing-value and outlier handling rules.",
        "Encode categorical variables and scale numeric features when needed.",
        "Split data into train/validation/test sets before modeling.",
        "Build baseline models and compare against more advanced models.",
    ]
    for step in steps:
        print(f"- {step}")
    return steps


def run_eda_report(
    df: pd.DataFrame,
    dataset_name: str = "Dataset",
    target_col: Optional[str] = None,
    date_col: Optional[str] = None,
    clean_data: bool = False,
    drop_duplicates: bool = False,
    standardize_column_names: bool = False,
    max_numeric_plots: int = 8,
    max_categorical_plots: int = 8,
) -> dict:
    """Run a full reusable EDA report and return the main outputs."""
    show_title("Exploratory Data Analysis (EDA) Report", level=1)
    show_title("Executive Summary")
    print("This report summarizes structure, quality, distributions, relationships, outliers, and modeling next steps.")

    show_title("Introduction")
    print(f"The analysis explores {dataset_name} to understand the dataset before deeper analysis or modeling.")

    working_df = data_loading_and_cleaning(
        df,
        drop_duplicates=drop_duplicates,
        standardize_column_names=standardize_column_names,
    ) if clean_data else df.copy()

    info = dataset_information(working_df, dataset_name=dataset_name, date_col=date_col)
    overview = data_overview(working_df)
    summaries = summary_statistics(working_df)
    missing = missing_values_analysis(working_df)
    plot_feature_distributions(
        working_df,
        max_numeric=max_numeric_plots,
        max_categorical=max_categorical_plots,
    )
    univariate = univariate_analysis(working_df)
    bivariate_analysis(working_df, target_col=target_col)
    engineered_df = feature_engineering(working_df, date_col=date_col)
    outliers = outlier_detection_iqr(engineered_df)
    corr = correlation_analysis(engineered_df)
    custom = custom_target_analysis(engineered_df, target_col=target_col)
    recommendations = conclusions_and_recommendations(missing, outliers, corr)
    interactive = interactive_eda_tools(engineered_df)
    steps = next_steps()

    return {
        "data": working_df,
        "engineered_data": engineered_df,
        "dataset_information": info,
        "overview": overview,
        "summaries": summaries,
        "missing_values": missing,
        "univariate": univariate,
        "outliers": outliers,
        "correlations": corr,
        "custom": custom,
        "recommendations": recommendations,
        "interactive_tools": interactive,
        "next_steps": steps,
    }


## Example Usage

In [ ]:
# Example for this project. Uncomment and run when needed.
# df = pd.read_csv("../data/raw/ObesityDataSet_raw_and_data_sinthetic.csv")
# eda_outputs = run_eda_report(
#     df,
#     dataset_name="Obesity Dataset",
#     target_col="NObeyesdad",
#     clean_data=False,
# )
